# xml: html처럼 태그 기반으로 자료를 저장한 포멧
* xml parser를 통해서 str을 xml로 변환하는 작업이 필요
* xml로 변환이 되면 태그 기반으로 자료를 찾아서 정리
* 태그에서 자료를 추출할때는 beautifulsoup이라는 라이브러리를 이용

In [1]:
import requests
import json
import pandas as pd

In [2]:
rent_dic = {}
i = 1

while i <= 20:
    url = 'http://apis.data.go.kr/B551408/rent-loan-rate-info/rate-list'
    payload = {'serviceKey' : "lwJZpRMDn/p4YZe0Hzrf92CTBVmSbWToaTJgutxn3oJ6kss3RSoaZtwPvQiQLKYowcKSOnj3h6G54NvRysgQGA==", 
               'numOfRows' : 100, 
               'pageNo' : i,
               'dataType': 'json'}  

    response = requests.get(url, params=payload)
    data = response.json()

    for item in data['body']['items']:
        for key, value in item.items():
            rent_dic.setdefault(key,[]).append(value)
    
    i += 1
        
        
rent_df = pd.DataFrame(rent_dic)
rent_df

,bssYmdStart,interest4_1,interest3_2,interest4_2,interest2_1,interest1_2,interest3_1,interest2_2,interest1_1,bssYmdEnd,organId,callCenter
0,20250310,0,0,0,0,0,0,0,0,20250316,산업은행,1588-1500
1,20250310,0,0,0,0,0,0,0,0,20250316,제주은행,1588-0079
2,20250310,0,0,0,0,0,0,0,0,20250316,SC은행,1588-1599
3,20250310,4.89,0,0,0,0,0,0,0,20250316,전북은행,1588-4477
4,20250310,0,0,0,0,0,0,0,0,20250316,수협은행,1588-1515
5,20250310,4.26,0,0,0,0,0,0,0,20250316,우리은행,1599-5000
6,20250310,4.64,0,0,0,0,0,0,0,20250316,경남은행,1588-8585
7,20250310,3.5,0,0,0,0,0,0,0,20250316,카카오뱅크,1599-3333
8,20250310,3.38,0,0,0,0,0,0,0,20250316,아이엠뱅크,1588-5050
9,20250310,4.18,0,0,0,0,0,0,0,20250316,하나은행,1599-1111


* xml로 데이터를 받으면 처음에는 단순 문자열로 받게 된다.
* 문자열을 xml 문서로 변환해야 함
* beautifulsoup을 이용해 변환
* beautifulsoup의 메서드인 select, select_one을 이용해 css 셀렉터 기반으로 데이터가 위치한 태그를 찾아서 내용 추출

In [29]:
url = 'http://apis.data.go.kr/B551408/rent-loan-rate-info/rate-list'
payload = {'serviceKey' : "lwJZpRMDn/p4YZe0Hzrf92CTBVmSbWToaTJgutxn3oJ6kss3RSoaZtwPvQiQLKYowcKSOnj3h6G54NvRysgQGA==", 
           'numOfRows' : 20, 
           'pageNo' : 1,
           'dataType': 'XML'}  

response = requests.get(url, params=payload)
response = response.text
response

'<response><header><resultCode>00</resultCode><resultMsg>정상</resultMsg></header><body><pageNo>1</pageNo><totalCount>18</totalCount><numOfRows>20</numOfRows><items><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>산업은행</organId><callCenter>1588-1500</callCenter></item><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>제주은행</organId><callCenter>1588-0079</callCenter></item><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</inte

# BeautifulSoup
* https://www.crummy.com/software/BeautifulSoup/bs4/doc.ko/ 참고
* pip install beautifulsoup4
* from bs4 import BeautifulSoup as bs
* find, find_all 함수 : xml, html에서 태그 기반으로 내용을 찾음
* select, select_one 함수 : xml, html에서 css selector 기반으로 내용을 찾음
* find_all, select는 해당 태그나 css select를 가진 부분을 모두 찾아서 리스트로 반환
* find, select_one은 여러 태그나 css selector 중에서 가장 먼저 나오는 태그, selector를 한 개만 찾아줌
* 찾아온 태그.name : 태그 이름 반환
* 찾아온 태그.text, .string : 태그 안쪽의 텍스트 반환

### 반복문으로 태그를 자동 추출해서 만들기

In [4]:
from bs4 import BeautifulSoup as bs

In [30]:
soup = bs(response, 'xml')
print(soup)

<?xml version="1.0" encoding="utf-8"?>
<response><header><resultCode>00</resultCode><resultMsg>정상</resultMsg></header><body><pageNo>1</pageNo><totalCount>18</totalCount><numOfRows>20</numOfRows><items><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>산업은행</organId><callCenter>1588-1500</callCenter></item><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>제주은행</organId><callCenter>1588-0079</callCenter></item><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3

In [31]:
soup.select_one('resultCode')

<resultCode>00</resultCode>

In [32]:
items = soup.select_one('items')
item = items.select('item')

In [33]:
for tag in items.select_one('item'):
    print(tag.name)

bssYmdStart
interest4_1
interest3_2
interest4_2
interest2_1
interest1_2
interest3_1
interest2_2
interest1_1
bssYmdEnd
organId
callCenter


In [34]:
items = soup.select_one('items')
bssYmdStart_list = items.select('bssYmdStart')
interest4_1_list = items.select('interest4_1')
interest3_2_list = items.select('interest3_2')
interest4_2_list = items.select('interest4_2')
interest2_1_list = items.select('interest2_1')
interest1_2_list = items.select('interest1_2')
interest3_1_list = items.select('interest3_1')
interest2_2_list = items.select('interest2_2')
interest1_1_list = items.select('interest1_1')
bssYmdEnd_list = items.select('bssYmdEnd')
organId_list = items.select('organId')
callCenter_list = items.select('callCenter')

In [35]:
tag_list = [bssYmdStart_list,interest4_1_list, interest3_2_list, interest4_2_list,interest2_1_list, interest1_2_list, interest3_1_list, interest2_2_list, interest1_1_list, bssYmdEnd_list, organId_list, callCenter_list]

result = {}

for lists in tag_list:
    for item in lists:
        result.setdefault(item.name, []).append(item.text)

result

{'bssYmdStart': ['20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310'],
 'interest4_1': ['0',
  '0',
  '0',
  '4.89',
  '0',
  '4.26',
  '4.64',
  '3.5',
  '3.38',
  '4.18',
  '3.74',
  '3.7',
  '3.89',
  '3.84',
  '4.05',
  '4.0',
  '4.11',
  '4.07'],
 'interest3_2': ['0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0'],
 'interest4_2': ['0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0'],
 'interest2_1': ['0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0'],
 'interest1_2': ['0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0'

In [36]:
for item in soup.select("items"):
    for tags in item.select('item'):
        for tag in tags:
            result.setdefault(tag.name,[]).append(tag.text)
            
df = pd.DataFrame(result)
df

,bssYmdStart,interest4_1,interest3_2,interest4_2,interest2_1,interest1_2,interest3_1,interest2_2,interest1_1,bssYmdEnd,organId,callCenter
0,20250310,0,0,0,0,0,0,0,0,20250316,산업은행,1588-1500
1,20250310,0,0,0,0,0,0,0,0,20250316,제주은행,1588-0079
2,20250310,0,0,0,0,0,0,0,0,20250316,SC은행,1588-1599
3,20250310,4.89,0,0,0,0,0,0,0,20250316,전북은행,1588-4477
4,20250310,0,0,0,0,0,0,0,0,20250316,수협은행,1588-1515
5,20250310,4.26,0,0,0,0,0,0,0,20250316,우리은행,1599-5000
6,20250310,4.64,0,0,0,0,0,0,0,20250316,경남은행,1588-8585
7,20250310,3.5,0,0,0,0,0,0,0,20250316,카카오뱅크,1599-3333
8,20250310,3.38,0,0,0,0,0,0,0,20250316,아이엠뱅크,1588-5050
9,20250310,4.18,0,0,0,0,0,0,0,20250316,하나은행,1599-1111


In [12]:
for tag in soup.select('items')[0]:
    for key in tag:
        print(key.name, key.text)

bssYmdStart 20250310
interest4_1 0
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId 산업은행
callCenter 1588-1500
bssYmdStart 20250310
interest4_1 0
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId 제주은행
callCenter 1588-0079
bssYmdStart 20250310
interest4_1 0
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId SC은행
callCenter 1588-1599
bssYmdStart 20250310
interest4_1 4.89
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId 전북은행
callCenter 1588-4477
bssYmdStart 20250310
interest4_1 0
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId 수협은행
callCenter 1588-1515
bssYmdStart 20250310
interest4_1 4.26
interest3_2 0
interest4_2 0
i

## xml에 있는 모든 key를 추출하기

In [13]:
tag_names = []

for item in soup.select("item"):
    for tag in item:
        tag_names.append(tag.name)
tag_names

['bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2

# 집합자료형 set : 중복 없는 데이터 집합 만들기
* 리스트 속 중복된 요소들이 한개씩만 남게됨

In [14]:
tags = set(tag_names)
tags

{'bssYmdEnd',
 'bssYmdStart',
 'callCenter',
 'interest1_1',
 'interest1_2',
 'interest2_1',
 'interest2_2',
 'interest3_1',
 'interest3_2',
 'interest4_1',
 'interest4_2',
 'organId'}

In [15]:
result = {}
for tags in soup.select("item"):
    for tag in tags:
        result.setdefault(tag,[]).append()

TypeError: list.append() takes exactly one argument (0 given)

# 금융위원회_서민금융상품기본정보

In [ ]:
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo

In [ ]:
lwJZpRMDn/p4YZe0Hzrf92CTBVmSbWToaTJgutxn3oJ6kss3RSoaZtwPvQiQLKYowcKSOnj3h6G54NvRysgQGA==

## JSON 데이터 정리

In [16]:
result_dic = {}
i = 1

while i <= 48:
    url = 'https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo'
    payload = {'serviceKey' : "lwJZpRMDn/p4YZe0Hzrf92CTBVmSbWToaTJgutxn3oJ6kss3RSoaZtwPvQiQLKYowcKSOnj3h6G54NvRysgQGA==", 
               'numOfRows' : 100, 
               'pageNo' : i,
               'resultType': 'json'}  

    response = requests.get(url, params=payload)
    data = response.json()

    for item in data['response']['body']['items']['item']:
        for key, value in item.items():
            result_dic.setdefault(key,[]).append(value)
    
    i += 1
        
        
result_df = pd.DataFrame(result_dic)
result_df

,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202503,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
1,202503,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700
2,202503,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
3,202503,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
4,202503,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4736,202110,425,"온(溫, ON)택트 특례보증",2850만원,변동금리,대출실행 은행에 문의,5년,1년,4년,원금균등분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700
4737,202110,426,위기업종 지원 협약보증,5000만원,-,-,"1, 5년",1년,4년,원금균등분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700
4738,202110,427,익산시 소상공인 특례보증(2023 하나은행 출연),5000만원,고정금리,약 1.6%,5년,1년,4년,원금균등분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700
4739,202110,428,중소벤처기업부 소상공인자금,7000만원,자금별 별도금리 적용,-,5년,2년,3년,분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700


In [77]:
soup = bs(response2, 'xml')
print(soup)

<?xml version="1.0" encoding="utf-8"?>
<response>
<header>
<resultCode>00</resultCode>
<resultMsg>NORMAL SERVICE.</resultMsg>
</header>
<body>
<numOfRows>100</numOfRows>
<pageNo>48</pageNo>
<totalCount>4741</totalCount>
<items>
<item>
<basYm>202110</basYm>
<snq>389</snq>
<finPrdNm>경기도 중소기업 육성자금(운전_일자리창출기업)</finPrdNm>
<lnLmt>30000만원</lnLmt>
<irtCtg>-</irtCtg>
<irt>은행금리</irt>
<maxTotLnTrm>5년</maxTotLnTrm>
<maxDfrmTrm>2년</maxDfrmTrm>
<maxRdptTrm>3년</maxRdptTrm>
<rdptMthd>균등분할상환</rdptMthd>
<usge>운전</usge>
<trgt>중소기업</trgt>
<instCtg>일반기관</instCtg>
<ofrInstNm>경기신용보증재단</ofrInstNm>
<rsdAreaPamtEqltIstm>경기</rsdAreaPamtEqltIstm>
<suprTgtDtlCond>도내 중소기업으로서 아래 어느 하나에 해당하는 기업일자리기업 : 최근 1년 이내 신규고용을 창출한 기업(기존대비 고용인원 증가 및 1년 이상 고용유지를 확약한 경우에 한함)</suprTgtDtlCond>
<age>없음</age>
<incm>없음</incm>
<rsdArea xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:nil="true"/>
<crdtSc xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:nil="true"/>
<anin xmlns:xsi="http://www.w3.org/2001/XMLSchema-inst

## XML 데이터 정리

In [76]:
result2_dic = {}
i = 1

while i <= 48:
    url = 'https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo'
    payload = {'serviceKey' : "lwJZpRMDn/p4YZe0Hzrf92CTBVmSbWToaTJgutxn3oJ6kss3RSoaZtwPvQiQLKYowcKSOnj3h6G54NvRysgQGA==", 
               'numOfRows' : 100, 
               'pageNo' : i,
               'resultType': 'xml'}  

    response2 = requests.get(url, params=payload)
    response2 = response2.text

    soup2 = bs(response2, 'xml')

    for item in soup2.select('item'):
        for tag in item:
            if (tag.name):
                result2_dic.setdefault(tag.name, []).append(tag.text)
                
    i += 1

    
result2_df = pd.DataFrame(result2_dic)
result2_df

,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202503,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
1,202503,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700
2,202503,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
3,202503,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
4,202503,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4736,202110,425,"온(溫, ON)택트 특례보증",2850만원,변동금리,대출실행 은행에 문의,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4737,202110,426,위기업종 지원 협약보증,5000만원,-,-,"1, 5년",1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4738,202110,427,익산시 소상공인 특례보증(2023 하나은행 출연),5000만원,고정금리,약 1.6%,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4739,202110,428,중소벤처기업부 소상공인자금,7000만원,자금별 별도금리 적용,-,5년,2년,3년,분할상환,...,Y,,1,대출상품,,,,,,202502010700


In [75]:
result2_dic = {}
column = ['기준년월','순번', '금융상품명', '대출한도', '금리구분', '금리', '(최대)총대출기간', '(최대)거치기간', '(최대)상환기간', '상환방법', '용도', '대상', '기관구분', '제공기관명', '거주지역원금균등분할상환', '지원대상상세조건' ,'연령' ,'소득', '거주지역' ,'신용등급' ,'연소득' ,'주택보유수', '주택면적', '대출대상주택' ,'문의처 및 연락처' ,'보증기관', '가입(신청)방법', '중도상환수수료', '대출부대비용', '연체이자율(연)', '우대금리/가산금리 조건' ,'기타 참고사항', '취급기관', '연락처', '관련 사이트', '대상_필터', '취급기간_상세보기용', '상품존재여부', '문의처(코로나)', '상품구분', '상품명', '코로나19 지원구분', '코로나19 지원내용', '코로나19 지원대상 상세내용', '운영기한', '상품구분 2', '파일작성일시']

i = 1

while i <= 48:
    url = 'https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo'
    payload = {'serviceKey' : "lwJZpRMDn/p4YZe0Hzrf92CTBVmSbWToaTJgutxn3oJ6kss3RSoaZtwPvQiQLKYowcKSOnj3h6G54NvRysgQGA==", 
               'numOfRows' : 100, 
               'pageNo' : i,
               'resultType': 'xml'}  

    response2 = requests.get(url, params=payload)
    response2 = response2.text

    soup2 = bs(response2, 'xml')
                
    for item in soup2.select('item'):
        tags = item.find_all(True)
    
        for col, tag in zip(column, tags):
            result2_dic.setdefault(col, []).append(tag.text)
                
    i += 1
                
result2_df = pd.DataFrame(result2_dic)
result2_df

,기준년월,순번,금융상품명,대출한도,금리구분,금리,(최대)총대출기간,(최대)거치기간,(최대)상환기간,상환방법,...,상품존재여부,문의처(코로나),상품구분,상품명,코로나19 지원구분,코로나19 지원내용,코로나19 지원대상 상세내용,운영기한,상품구분 2,파일작성일시
0,202503,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
1,202503,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700
2,202503,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
3,202503,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
4,202503,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4736,202110,425,"온(溫, ON)택트 특례보증",2850만원,변동금리,대출실행 은행에 문의,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4737,202110,426,위기업종 지원 협약보증,5000만원,-,-,"1, 5년",1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4738,202110,427,익산시 소상공인 특례보증(2023 하나은행 출연),5000만원,고정금리,약 1.6%,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4739,202110,428,중소벤처기업부 소상공인자금,7000만원,자금별 별도금리 적용,-,5년,2년,3년,분할상환,...,Y,,1,대출상품,,,,,,202502010700


In [74]:
result3_dic = {}
column = ['기준년월','순번', '금융상품명', '대출한도', '금리구분', '금리', '(최대)총대출기간', '(최대)거치기간', '(최대)상환기간', '상환방법', '용도', '대상', '기관구분', '제공기관명', '거주지역원금균등분할상환', '지원대상상세조건' ,'연령' ,'소득', '거주지역' ,'신용등급' ,'연소득' ,'주택보유수', '주택면적', '대출대상주택' ,'문의처 및 연락처' ,'보증기관', '가입(신청)방법', '중도상환수수료', '대출부대비용', '연체이자율(연)', '우대금리/가산금리 조건' ,'기타 참고사항', '취급기관', '연락처', '관련 사이트', '대상_필터', '취급기간_상세보기용', '상품존재여부', '문의처(코로나)', '상품구분', '상품명', '코로나19 지원구분', '코로나19 지원내용', '코로나19 지원대상 상세내용', '운영기한', '상품구분 2', '파일작성일시']


for item in soup2.select('item'):
    tags = item.find_all(True)
    
    for col, tag in zip(column, tags):
        result3_dic.setdefault(col, []).append(tag.text)
            
result3_df = pd.DataFrame(result3_dic)
result3_df

,기준년월,순번,금융상품명,대출한도,금리구분,금리,(최대)총대출기간,(최대)거치기간,(최대)상환기간,상환방법,...,상품존재여부,문의처(코로나),상품구분,상품명,코로나19 지원구분,코로나19 지원내용,코로나19 지원대상 상세내용,운영기한,상품구분 2,파일작성일시
0,202110,389,경기도 중소기업 육성자금(운전_일자리창출기업),30000만원,-,은행금리,5년,2년,3년,균등분할상환,...,Y,,1,대출상품,,,,연중 수시(자금 소진시까지),정책자금,202502010700
1,202110,390,경기도 중소기업 육성자금(운전_혁신창업사업화기업),50000만원,고정금리,은행금리,5년,2년,3년,균등분할상환,...,Y,,1,대출상품,,,,연중 수시(자금 소진시까지),정책자금,202502010700
2,202110,391,경기도 중소기업 육성자금(운전_성장디딤돌 기업),50000만원,-,은행금리,5년,2년,3년,균등분할상환,...,Y,,1,대출상품,,,,연중 수시(자금 소진시까지),정책자금,202502010700
3,202110,392,금융기관 특별출연 협약보증,80000만원,-,-,5(시설자금: 5년 초과 가능),-,-,-,...,Y,,1,대출상품,,,,2011.03.14 ~ 자금소진시 또는 사업종료시,정책자금,202502010700
4,202110,393,경기도 취약 소상공인 자금(경영개선자금),2000만원,-,4.15%,5년,0년,5년,원리금균등분할상환,...,N,,1,대출상품,,,,2022.04.11 ~ 자금 소진 시까지,정책자금,202502010700
5,202110,394,경기도 취약 소상공인 자금(특별지원자금),5000만원,-,4.15%,5년,0년,5년,원리금균등분할상환,...,N,,1,대출상품,,,,2022.04.11 ~ 자금 소진 시까지,정책자금,202502010700
6,202110,395,일반경영안정자금(창업초기자금),7000만원,,-,5년,2년,3년,,...,N,,1,대출상품,,,,상시,정책자금,202502010700
7,202110,396,일반경영안정자금(여성가장 지원),7000만원,,-,5년,2년,3년,,...,N,,1,대출상품,,,,상시,정책자금,202502010700
8,202110,397,일반경영안정자금(사업전환자금),10000만원,,-,5년,2년,3년,,...,N,,1,대출상품,,,,상시,정책자금,202502010700
9,202110,398,북구 디딤돌 특례보증,2000만원,,-,5년,1년,4년,원금균등분할상환,...,N,,1,대출상품,,,,2022.02.18.~ 소진시,정책자금,202502010700


In [83]:
for item in soup2.select("item"):
    for tags in item:
        print(tags.name)
        # 개행이 None으로 출력되어 예외처리 필요

None
basYm
None
snq
None
finPrdNm
None
lnLmt
None
irtCtg
None
irt
None
maxTotLnTrm
None
maxDfrmTrm
None
maxRdptTrm
None
rdptMthd
None
usge
None
trgt
None
instCtg
None
ofrInstNm
None
rsdAreaPamtEqltIstm
None
suprTgtDtlCond
None
age
None
incm
None
rsdArea
None
crdtSc
None
anin
None
housHoldCnt
None
housAr
None
lnTgtHous
None
rfrcCnpl
None
grnInst
None
jnMthd
None
rpymdCfe
None
lnIcdcst
None
ovItrYr
None
prftAddIrtCond
None
etcRefSbjc
None
hdlInst
None
cnpl
None
rltSite
None
tgtFltr
None
hdlInstDtlVw
None
prdExisYn
None
cv19Rfrc
None
prdCtg
None
prdNm
None
cv19SuprCtg
None
cv19SuprCtn
None
cv19SuprTgtDtlCtn
None
mgmDln
None
prdCtg2
None
fileWrtDt
None
None
basYm
None
snq
None
finPrdNm
None
lnLmt
None
irtCtg
None
irt
None
maxTotLnTrm
None
maxDfrmTrm
None
maxRdptTrm
None
rdptMthd
None
usge
None
trgt
None
instCtg
None
ofrInstNm
None
rsdAreaPamtEqltIstm
None
suprTgtDtlCond
None
age
None
incm
None
rsdArea
None
crdtSc
None
anin
None
housHoldCnt
None
housAr
None
lnTgtHous
None
rfrcCnpl
None
grnI